# RepairScope-X — Notebook 13
## Independent Dream Self-Repair Confirmation

Notebook 12 established the first direct Dream→Dream self-repair surface:

- **240/240** self-repair trials completed;
- **51 task-successful repairs**;
- **36** successes came from selective-remasking conditions;
- all **3 datasets** showed successful self-repair;
- all **3 selective scopes** produced at least one success;
- best fixed scope = `open_last_75` at **26.67%** task success;
- matched-NFE `global` = **25.00%** task success;
- per-instance oracle = **35.00%**;
- Oracle Repair Headroom = **8.33 percentage points**;
- `DREAM_SELF_REPAIR_SURFACE_READY = true`.

Notebook 12 also exposed a preservation–recurrence pattern:

- `open_last_25`: preservation ≈ **87.0%**, same-wrong recurrence given failure ≈ **74.0%**;
- `open_last_75`: preservation ≈ **47.3%**, recurrence ≈ **43.2%**;
- `global`: preservation ≈ **33.8%**, recurrence ≈ **57.8%**.

The individual selective-vs-global success contrasts were **not statistically significant**, so this notebook does not assume selective remasking is superior. It independently tests whether the self-repair and preservation/hysteresis patterns reproduce.

## Cohort design

### Primary confirmation set
Eight Dream-origin failures from **8 problem IDs never used in Notebook 12**:

- 4 held-out Countdown problems;
- 4 held-out GSM8K problems.

### Secondary SVAMP set
Notebook 11 produced no additional failed SVAMP problem IDs outside Notebook 12. It did contain **two unused failed Dream trajectories** from persistent SVAMP problems. These are included only as secondary cross-dataset support and are never counted as unseen-problem confirmation.

## Frozen actions

To avoid another broad exploratory grid, Notebook 13 confirms only the three most informative scopes:

- `open_last_25` — high-preservation / high-recurrence condition;
- `open_last_75` — Notebook 12 best fixed selective scope;
- `global` — matched-64-NFE fresh regeneration control.

Five new repair seeds:

`3501, 3502, 3503, 3504, 3505`

Total:

**10 failures × 3 actions × 5 seeds = 150 independent repair generations**

Primary unseen-problem analysis uses:

**8 failures × 3 actions × 5 seeds = 120 trials**

The 30 SVAMP trials are reported separately and in an all-data descriptive table.

Embedded payload SHA-256: `789310b6f54e30c4872dd71e28df461b712445282a3631c6c007fed93b9fd3e2`

In [ ]:
!pip -q install -U "transformers>=4.57,<5" "accelerate>=1.0" "bitsandbytes>=0.47" pandas scipy

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, ast as pyast
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import binomtest

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 3500
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Notebook 13 requires a CUDA GPU runtime."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", GPU_NAME)
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if GPU_GB < 14:
    raise RuntimeError(
        "This exploratory NF4 Dream confirmation expects a T4-class GPU (~15 GB) or larger."
    )

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError(
        "GPU memory is already occupied. Restart the runtime and run from the top."
    )

print("Runtime preflight: PASS")

## 2. Recover and validate the frozen confirmation cohort

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "789310b6f54e30c4872dd71e28df461b712445282a3631c6c007fed93b9fd3e2"

_raw = zlib.decompress(
    base64.b64decode('''eNrlWwuT08gR/itTLlKXgDHz0ssVkiIB6rgilyvg6urqlhJjaWwrq4dPjzUbjv+e7hnZlmTvxrtechCWwqxHPf2er3tGw4dRVOTzpMzCuUrSptTVaPrLh9FC57pUdVLkYRKPpkDU5HVcrPNppNJkZh9NmU+dKfN+i0utst+EoGw0Hq3KYpbq7Ih5QByrWlW67lKGc1BiiYxUWSdIj093s62IbIWTfqw0qZea5E0202VFmDsmjI6JymPCfaLfq6hOL0mRR0BXEJgXN5GdUqtyoWsi5IS80nVT5kSRC5ASE1Um9TLTdRIBgxUoU4FUw1LD3zU8JO+ev/j+ycvwyfevf3r2akr+bOX/5d0ElLOMR1PgTMejSmvwA7pmPCqaetWg2m8KUhXpBSqSVKR12JiswRAgR02bOzHsLD/Lv6lqMzwv0rRYJ/nisH1TpGUueQBi4ANkPAYWOPgGdew4AlSD7ypatsr1lUEFW22qgTrI7HUxNqPXeTqpjDLGxe50SrpagX8vdJnMk8jmwvTDqKrLJKpDmB7Dv2Ywb9J0PAL1gDTqPeE+BmUQIsugaiLQGdJ/rtJK76YPxyFhz/cGd9kL3JoI8gnD/uEwsTE+3Fm8UXg3EgJJozfjVdGUkQ7bXID1yVzQm1H4QHveDg2Kljo613EYQTASWGHtmu4KHA2dauU5Zn6a1LD404OSMPobTUKM+GgKBuuWQxgVZamjujX143ggVcidLDGUhQOHJez81hWBcj++/fjxbiIIOdyoNJwX5VqV6LsUlXLleIOF1qEXqgrr4lzn8DBwx6M10IWVBhCNYYTTCRecU+l6jutJCbNXWp2HF6XKwsUMHDzxAk8GrmC+zzwmgvGobCBpkkWuMGkQ6mZCxZ5PA18zb+5J6nERBDSSszmP5tLxfGfuMhVF3OUz3/VixpyZS2euH819n+sR+gOYgTqj4hy+6bIsyk0utSh/Y1Rvq4Sqi/IyLIsUNV2VSabga5MDyuVhi2MjjPoN6od/u/Lhf4LqITsgK44BWfapige7rnYcUgRgGbH9SHNMrYFs3taZY6rD2dkv5OyMSfKwBQ4Bv8HHY5B+dvYWSV7q2rCukmyVJvNLw/sAm2uZbGqEyqs1lJe2HMC0vt/Y21OLAevWAnaTUmAh73OoBHKLz6JXCdjRhWAYih1C7xWDnrCNxKPx+nBJYDt5bCjvU9eDa8J4d+XAcXxfMM5kQKV/sB5wJqjnOdJhrvu51gP/f1UOBLtdPRDs7guCAPTkLYLCEjmiIEj6qSqCpDcsCZIeqAnXWHTbmkAnDiB3nWTQ8P9RGCChprf8E4C6pC2o4y5iTJbFWgNkj+0mIi9qkJxeXqPhmMyamsCSLDpEBEROyHqZwB4ksWyS3DxPk6qenFoWgPue27+4LYIwME1bwH47NOiYyoCBvT8Iaad9pwO0RrftST2xOEjaqUZDgZJ+8fsFFkyECETApBsEjvMlbxiGuP3pKgS/5Y6Bf4ItQwAttte22EcVCId/sgrhcHrCidM8AVmA/oeVqZeqtqB9pPU4BpObtK424NzxAFaEf+aarApgD6qgck3dOfwJAHmYBx/A9zFOOVBDEquMqSDb+UbRro6HdcU6t41OQVCvvoo9a04uKW1ohpH60nYbAQK710L82z17jikqncB2a4k7BPehqNPOnZyutFbbjjCH0/+HrQZzOWWOFD71Al98sZWE3+1eY1Fl/jn4XyX51HH4dW8rhqS9emEeHlclflIp5BaZ6zQlcxi0xyDJYglIvkpVDaHOSDEH1MoA+iI1h1Y7No02oJOZtVJVTZ6qC6wOc8MLoo0UkgA2Q3pOyIs5wSbdyChKA91xbBRTKQgstSZZUQKmmc7c4GKsV6DBTM9xfJnU9WZ7sCghHLFFR/y+UbKCcgCaX2iDsSsVQa2AmoKIvjT2QAlBxluj1qpq1S7yv3Zqk3tlZer6amPcRo98YJXYM2a82wYIABWJ50h86yI8ScLiU9QqHW+9JbGV5UjpbihfJ3lbpq+N2bjnnN0vMGiNRvNNETOEPkzFw5WNGAIyeV+73jScwzmYPtmyPrnycN4tPO6n3cygGp/ZcbwrA+l50me+5PRAd+1T6jIWeNynvi/83xUTr0eqO4RBj1F5JAwi6W1h8AnB3Dg3zVakyvISOt5MvU+yxqAfE2OHUnK+qAwWNqtVmmhYFz/rChZIrC7tcrM8cImkhYp12xAzmDkr3m/AYI0r1QwiO3w7OSFvAEBg/dcH5pIIM6hqwcNOdntzCyO/P1ktwCOWhUNJpaLzPgcYPE/SYgHZteHzbbEGk/NLMlPA25AZFB9SGh/hwXmdAP6gvG3XjOZ3oZRd0+Vv3bUErZUFPmANwJ3Ul32fT8hPu+YfAhg1ADntVsVMM8rWOAnHNuEhKoW8iS+3PgG4qpoZJEtUo7u20DmQvdlhYPVYolMaMNrUJ7R8plt+ZmfAJuTvPYX6qpiw7+J/fxP2RwjpdEztt7OcX8/GpsAuF+63GQBsTgVdt9ftM/qVoS6b+CIQUlAPUFP4vncAdhn3Au4BrjKHCe+zgd19ZLzL7tPzju4+gfS2sPscukHTyqgsgbat1E0OFGPyA/Yd2JvMmksEIehLVtj2GfidFbl+CJATAZ6da3v6sFTp3MKryjDzTNunslkDeY0NDPlWf1ORRWG6yALZEm7X9I7vsoBOrVgYmD402xzt9lhUuLsH6KiSeIvuCIoWY1ud9EaljghkuYHbnQLzoogtA4SO5rILpSK4Ekrv2Flj6PVBgW86/B5hC+p3XbVzzgmO3ZPk4/GtgcYDsydDObf2/r6F9BE/uXvt3dYJvjoc5T60rQI6VOFR3znUvTqSe47DuBs4gn0+3ese0t0pjHL3aBjl7m1hFJspM7ppGHY5n8H6v4TmolRj8p3K8dIDQsB3GnvcpbrQCBxy6sB+C7fpu3ZoyAKe7Cbc85zxoBNLi6oycix/FGWXY6bOYc7cMoiKbJbkZkZSbjjb5VyUKl9ou6GucGsLNBmBRe3TP2z7KbteIRr2pHOz3oLr+0xjkHUQWvvAeeA+Zs6EPMN2Fv274b+lWRcltM5g5iPmACDdcybWNmxWpZnSogh8vyfvAw0ellurccwZ0jiGxjGt/rjVCIT2HQaIZF2CLjbvse5xI1469obkmJjD9akZofRUzEImQyd+TZhFPY9z4UAH6HDJDvV+VMB2m1HhCg9g6zMCrSGunAha1YXKVi3zCErnQ2iFdwL4ELgOkffAyxAcB15PVZ4jMKSpjmooyUVdp7jVWkGX9K02x4NrotbqkribZVqkeGiHWGHfhYCUc9zewjxzGQBaB9zhdljB9nFtpkzI99B4ARWuSrdPg20QrLxWFVCzsyk2LU2XOE5iEhvV7euojiamzamXRWXU2RnQ7edk/8o170CWdQikRVlvjgHM6Wt7o8A0dVs9xiQ1uIFrADe17/ec1nY7YHBM3pOH4MVUz2t75tBzF5q4T/wAHz2GLw8I6NzGpedDM8c+B0LYULYnDZlWeQUP8PQSjxCJNQwnQYt6veNPvi/ndpFNyq8P2qjPAhG4ASCF4/JD0MYp9+EZpY7v/L5vWP47+hyCN2suAhws7hBm/wuSB59XKtM3Q7tABjdBOyC/Ldq9bKAbwsM5WNpM4k4tye17Z3O2Zl5u4BJXQJTpzauTVaoucYLjWYrKkkC+XSSWsrufNBwRn/AwLELgag/okLALQpwH9tD9IA69ac/Bdj3hDoFaIcYaIyLuvaHPmrROVr3rUp1Zy60H+mbPhvStrVsHmCO3N0aVltVjRJetGx9ZPvc7jsIZgFYdmByeHk4PMEWWhsveE87Hvstxyru2GdvedLCP3p3+GgS4dNFrF6Wv6W0IwJILWzNKKWwbuXPoFbHjBa7vMeY7gorPC8D2AOUEAHsLEShqPSuKc8bDBXjQXJsoE3MjgEvIjCZPfm10aMZAo81wN0DzJg1LvVIJXo9wGK55U2wvYNpVZMLdolwVYiMS9kMuulyUyd+Wrksy01UdzpP3EHjV5vioWEFDmip44DmjHklfAp1wd/ADCFWqKNV7lMKhvR8QHadZ2FJbo8KlVnFZFBlOoL7o/cAGINMQvhyWboprstLlhS0bEP/2XkSW5PhWKGzzPJ9rm92/Niqvk3+3a340y2fwTIZyltR4jyUFNTC2YOzTV8+e/CN8/ezl8/DVsx+evHgVvv7x1fMnf38GX588/bm9stELuvUbGJthQ2+uqmxduUiLmUoR1duMcPegIixN0tAJc/ZW8/aZ7LvZ66fPlgyXosnPdVnkCzMcwto1HPruFOhOlQ88hUBmhrv+NeET5gDeCbhHBcMXoIZs7xaiZEy4VMC2DeKNVOp9eKC58b0AN25CcIfb1YrJ1bpI9o1YQP7mm/9Vam7Vev0fbCIO5C53jnT7XgpfFQRxhdf3OVwRA5cNo3iTGPgeoC3EQPrQEgbsyhgwlwLoQodJHS4873AQJHM94AMILISU3l4QHHp9EDx5hdsdepzbuT/Mxytz/1S3Oyelvge5D0VOCi6h9jP3KrdDa+D5Urhceg71D3sdAiI4d3zOpfQ9cWOvu1d53Ts22dmxyS6dqzDmWLeLPVk3cbv0hAxgC0RFwPzAvdLtgkFEA88PPOqjzofcDsKphwfcQCRxOzVEHHm926VgfvcPVv22tiLw2/4cAA9qGrYVYVbEOrXNx1MceFiUi0f2twv68EVuu7uH3t9G/Rmlvkg2l+YhaWU0C+ZKzT1XuAGPo3kQQ9J4kSv5TPMZV67So4//AS24Qtk=''')
)

assert hashlib.sha256(_raw).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(_raw.decode("utf-8"))
FAILURES = UPSTREAM["confirm_failures"]
UPSTREAM_GATE = UPSTREAM["notebook12_gate"]
UPSTREAM_ACTIONS = pd.DataFrame(
    UPSTREAM["notebook12_action_summary"]
)

assert len(FAILURES) == 10
assert Counter(x["dataset"] for x in FAILURES) == {
    "countdown_fresh": 4,
    "gsm8k": 4,
    "svamp": 2,
}
assert Counter(x["confirmatory_role"] for x in FAILURES) == {
    "primary_unseen_problem": 8,
    "secondary_new_trajectory_same_problem": 2,
}
assert all(not x["task_success"] for x in FAILURES)
assert UPSTREAM_GATE["DREAM_SELF_REPAIR_SURFACE_READY"] is True

PRIMARY_FAILURE_IDS = {
    x["failure_id"]
    for x in FAILURES
    if x["confirmatory_role"] == "primary_unseen_problem"
}

SECONDARY_FAILURE_IDS = {
    x["failure_id"]
    for x in FAILURES
    if x["confirmatory_role"] == "secondary_new_trajectory_same_problem"
}

assert len(PRIMARY_FAILURE_IDS) == 8
assert len(SECONDARY_FAILURE_IDS) == 2

print("Payload SHA-256:", EMBEDDED_PAYLOAD_SHA256)
print("Total failures:", len(FAILURES))
print("Primary unseen-problem failures:", len(PRIMARY_FAILURE_IDS))
print("Secondary SVAMP trajectories:", len(SECONDARY_FAILURE_IDS))
print("Dataset distribution:", dict(Counter(x["dataset"] for x in FAILURES)))

for f in FAILURES:
    print(
        f["failure_id"],
        "|",
        f["dataset"],
        "|",
        f["confirmatory_role"],
    )

print("Confirmation cohort validation: PASS")

## 3. Strict + numeric verifier

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)


def close_num(a, b):
    return (
        a is not None
        and math.isclose(
            float(a),
            float(b),
            abs_tol=1e-8,
            rel_tol=1e-8,
        )
    )


def strict_final(text):
    vals = FINAL_RE.findall(str(text))

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def last_number(text):
    vals = NUM_RE.findall(str(text).replace(",", ""))

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def basic_verification(text, target):
    strict = strict_final(text)
    numeric = last_number(text)
    target = float(target)

    return {
        "strict_prediction": strict,
        "numeric_prediction": numeric,
        "target": target,
        "strict_success": bool(close_num(strict, target)),
        "numeric_success": bool(close_num(numeric, target)),
    }


assert basic_verification("x\nFINAL_ANSWER: 4", 4)["strict_success"]
assert basic_verification("The result is 4.", 4)["numeric_success"]
assert not basic_verification("The result is 5.", 4)["numeric_success"]

print("Strict/numeric verifier tests: PASS")

## 4. Countdown task verifier

In [ ]:
COUNTDOWN_PROMPT_RE = re.compile(
    r"Use the numbers\s+(.+?)\s+exactly once\s+"
    r"to produce the target\s+"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+))",
    flags=re.IGNORECASE,
)

ARITH_RUN_RE = re.compile(
    r"[0-9eE+\-*/().=\s]{3,}"
)

ALLOWED_BINOPS = (
    pyast.Add,
    pyast.Sub,
    pyast.Mult,
    pyast.Div,
)

ALLOWED_UNARY = (
    pyast.UAdd,
    pyast.USub,
)


def parse_countdown_prompt(prompt):
    m = COUNTDOWN_PROMPT_RE.search(str(prompt))

    if m is None:
        raise ValueError("Countdown prompt format was not recognized.")

    source_nums = [
        float(x)
        for x in NUM_RE.findall(m.group(1))
    ]

    return source_nums, float(m.group(2))


def safe_arith_tree_value_and_literals(expr):
    try:
        tree = pyast.parse(expr, mode="eval")
    except Exception:
        return None

    literals = []

    def walk(node):
        if isinstance(node, pyast.Expression):
            return walk(node.body)

        if isinstance(node, pyast.Constant):
            if not isinstance(node.value, (int, float)):
                raise ValueError

            value = float(node.value)

            if not math.isfinite(value):
                raise ValueError

            literals.append(value)
            return value

        if isinstance(node, pyast.BinOp) and isinstance(node.op, ALLOWED_BINOPS):
            a = walk(node.left)
            b = walk(node.right)

            if isinstance(node.op, pyast.Add):
                return a + b
            if isinstance(node.op, pyast.Sub):
                return a - b
            if isinstance(node.op, pyast.Mult):
                return a * b
            if isinstance(node.op, pyast.Div):
                if abs(b) < 1e-15:
                    raise ZeroDivisionError
                return a / b

        if isinstance(node, pyast.UnaryOp) and isinstance(node.op, ALLOWED_UNARY):
            value = walk(node.operand)

            if isinstance(node.op, pyast.USub):
                return -value
            return value

        raise ValueError

    try:
        value = walk(tree)

        if not math.isfinite(value):
            return None

        return {
            "value": float(value),
            "literals": literals,
        }
    except Exception:
        return None


def number_multiset_equal(a, b):
    if len(a) != len(b):
        return False

    aa = sorted(float(x) for x in a)
    bb = sorted(float(x) for x in b)

    return all(
        close_num(x, y)
        for x, y in zip(aa, bb)
    )


def countdown_candidates(text):
    text = (
        str(text)
        .replace("×", "*")
        .replace("÷", "/")
        .replace("\\times", "*")
        .replace("\\div", "/")
    )

    candidates = []

    for m in ARITH_RUN_RE.finditer(text):
        run = m.group(0).strip()

        if not any(op in run for op in ["+", "-", "*", "/"]):
            continue

        candidates.extend(
            p.strip()
            for p in run.split("=")
            if p.strip()
        )

    return sorted(set(candidates), key=len, reverse=True)


def countdown_task_verification(text, prompt):
    source_nums, target = parse_countdown_prompt(prompt)

    for expr in countdown_candidates(text):
        parsed = safe_arith_tree_value_and_literals(expr)

        if parsed is None:
            continue

        if (
            number_multiset_equal(parsed["literals"], source_nums)
            and close_num(parsed["value"], target)
        ):
            return {
                "task_success": True,
                "valid_expression": expr,
                "expression_value": parsed["value"],
                "source_numbers": source_nums,
                "target": target,
            }

    return {
        "task_success": False,
        "valid_expression": None,
        "expression_value": None,
        "source_numbers": source_nums,
        "target": target,
    }


_demo_prompt = (
    "Use the numbers 19, 29, and 24 exactly once "
    "to produce the target 14. "
    "Return a valid arithmetic expression."
)

assert countdown_task_verification(
    "19 + 24 - 29 = 14.",
    _demo_prompt,
)["task_success"]

assert not countdown_task_verification(
    "29 - 19 = 10. FINAL_ANSWER: 14",
    _demo_prompt,
)["task_success"]

print("Countdown verifier tests: PASS")

## 5. Unified repair verifier

In [ ]:
def verify_repair(text, failure):
    basic = basic_verification(
        text,
        failure["target"],
    )

    if failure["dataset"] == "countdown_fresh":
        structured = countdown_task_verification(
            text,
            failure["prompt"],
        )
        task_success = bool(structured["task_success"])
    else:
        structured = None
        task_success = bool(basic["numeric_success"])

    return {
        **basic,
        "task_success": task_success,
        "countdown_structured": structured,
    }


for failure in FAILURES:
    starting = verify_repair(
        failure["output"],
        failure,
    )

    assert not starting["task_success"], (
        "A frozen starting failure unexpectedly passes: "
        + failure["failure_id"]
    )

print("All frozen starting responses remain failures: PASS")

## 6. Frozen confirmatory configuration

In [ ]:
MODEL_ID = "Dream-org/Dream-v0-Instruct-7B"
MODEL_REVISION = "05334cb9faaf763692dcf9d8737c642be2b2a6ae"

ACTIONS = [
    "open_last_25",
    "open_last_75",
    "global",
]

REPAIR_SEEDS = [
    3501,
    3502,
    3503,
    3504,
    3505,
]

DENOISING_STEPS = 64
TEMPERATURE = 0.2
TOP_P = 0.95
ALG = "entropy"
ALG_TEMP = 0.0

EXPECTED_ALL = (
    len(FAILURES)
    * len(ACTIONS)
    * len(REPAIR_SEEDS)
)

EXPECTED_PRIMARY = (
    len(PRIMARY_FAILURE_IDS)
    * len(ACTIONS)
    * len(REPAIR_SEEDS)
)

assert EXPECTED_ALL == 150
assert EXPECTED_PRIMARY == 120

print("Actions:", ACTIONS)
print("Repair seeds:", REPAIR_SEEDS)
print("Expected all-data trials:", EXPECTED_ALL)
print("Expected primary unseen-problem trials:", EXPECTED_PRIMARY)

## 7. Load Dream-7B in exploratory NF4 mode

In [ ]:
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
).eval()

assert hasattr(model, "diffusion_generate")

MODEL_DEVICE = next(model.parameters()).device

print("MODEL LOAD: PASS")
print("Class:", type(model).__name__)
print("Device:", MODEL_DEVICE)
print("Load seconds:", round(time.time() - t0, 2))
print("Allocated GB:", round(torch.cuda.memory_allocated()/2**30, 3))

## 8. Prompt/canvas helpers

In [ ]:
SYSTEM_PROMPT = (
    "Solve the quantitative problem carefully. "
    "Give a mathematically correct answer. "
    "When possible, end with `FINAL_ANSWER: <number>`."
)


def encode_problem_prompt(problem):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    return {
        "input_ids": encoded.input_ids.to(MODEL_DEVICE),
        "attention_mask": encoded.attention_mask.to(MODEL_DEVICE),
    }


def encode_failed_response(text):
    ids = tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids[0]

    if len(ids) < 4:
        raise RuntimeError(
            "Frozen Dream failure retokenized to fewer than four tokens."
        )

    return ids


def force_mask_for_action(length, action):
    mask = torch.zeros(
        length,
        dtype=torch.bool,
    )

    if action == "global":
        return mask

    preserve_fraction = {
        "open_last_25": 0.75,
        "open_last_75": 0.25,
    }[action]

    cut = max(
        1,
        int(math.floor(length * preserve_fraction)),
    )

    mask[:cut] = True
    return mask


for failure in FAILURES:
    ids = encode_failed_response(failure["output"])
    print(
        failure["failure_id"],
        "| tokens =",
        len(ids),
    )

print("Canvas preparation: PASS")

## 9. Actual forward-call counter

In [ ]:
class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(self, module, inputs):
        self.calls += 1

    def __enter__(self):
        self.calls = 0
        self.handle = self.module.register_forward_pre_hook(
            self._hook
        )
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

## 10. Dream self-repair runner

In [ ]:
def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def run_self_repair(
    failure,
    action,
    repair_seed,
):
    reseed(repair_seed)

    prompt = encode_problem_prompt(
        failure["prompt"]
    )

    original_ids = (
        encode_failed_response(
            failure["output"]
        )
        .to(MODEL_DEVICE)
    )

    response_len = int(
        original_ids.numel()
    )

    force_mask_cpu = force_mask_for_action(
        response_len,
        action,
    )

    force_mask = force_mask_cpu.to(
        MODEL_DEVICE
    )

    prompt_len = int(
        prompt["input_ids"].shape[1]
    )

    def clamp_hook(step, x, logits):
        generated = x[
            :,
            prompt_len:
            prompt_len + response_len
        ]

        if force_mask.any():
            generated[:, force_mask] = (
                original_ids[
                    force_mask
                ].unsqueeze(0)
            )

        return x

    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()

    with ForwardCounter(model) as fc:
        output = model.diffusion_generate(
            prompt["input_ids"],
            attention_mask=
                prompt["attention_mask"],
            max_new_tokens=
                response_len,
            output_history=False,
            return_dict_in_generate=True,
            steps=
                DENOISING_STEPS,
            temperature=
                TEMPERATURE,
            top_p=
                TOP_P,
            alg=
                ALG,
            alg_temp=
                ALG_TEMP,
            generation_tokens_hook_func=
                clamp_hook,
        )

    wall = time.time() - t0

    final_ids = output.sequences[
        0,
        prompt_len:
        prompt_len + response_len
    ].detach()

    if force_mask.any():
        assert torch.equal(
            final_ids[force_mask],
            original_ids[force_mask],
        )

    repaired = tokenizer.decode(
        final_ids.tolist(),
        skip_special_tokens=True,
    ).strip()

    verification = verify_repair(
        repaired,
        failure,
    )

    final_cpu = final_ids.cpu()
    original_cpu = original_ids.cpu()

    token_position_preservation = float(
        (
            final_cpu
            == original_cpu
        )
        .float()
        .mean()
        .item()
    )

    forced_fraction = float(
        force_mask_cpu
        .float()
        .mean()
        .item()
    )

    original_numeric = last_number(
        failure["output"]
    )

    repaired_numeric = verification[
        "numeric_prediction"
    ]

    same_wrong_numeric = bool(
        not verification["task_success"]
        and original_numeric is not None
        and repaired_numeric is not None
        and close_num(
            original_numeric,
            repaired_numeric,
        )
    )

    return {
        "failure_id":
            failure["failure_id"],
        "problem_id":
            failure["problem_id"],
        "dataset":
            failure["dataset"],
        "confirmatory_role":
            failure["confirmatory_role"],
        "source_generation_seed":
            int(failure["seed"]),
        "action":
            action,
        "repair_seed":
            int(repair_seed),
        "response_tokens":
            response_len,
        "forced_tokens":
            int(force_mask_cpu.sum()),
        "open_tokens":
            int((~force_mask_cpu).sum()),
        "forced_fraction":
            forced_fraction,
        "strict_success":
            bool(
                verification["strict_success"]
            ),
        "numeric_success":
            bool(
                verification["numeric_success"]
            ),
        "task_success":
            bool(
                verification["task_success"]
            ),
        "same_wrong_numeric":
            same_wrong_numeric,
        "verification":
            verification,
        "original_output":
            failure["output"],
        "repaired_output":
            repaired,
        "token_position_preservation":
            token_position_preservation,
        "actual_forward_calls":
            int(fc.calls),
        "wall_seconds":
            float(wall),
        "peak_vram_gb":
            float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
    }

## 11. One real confirmatory smoke test

In [ ]:
smoke = run_self_repair(
    FAILURES[0],
    "open_last_75",
    REPAIR_SEEDS[0],
)

print(
    json.dumps(
        {
            k: v
            for k, v in smoke.items()
            if k not in {
                "original_output",
                "repaired_output",
                "verification",
            }
        },
        indent=2,
    )
)

assert smoke["actual_forward_calls"] >= DENOISING_STEPS
assert math.isfinite(smoke["wall_seconds"])

print("CONFIRMATORY SELF-REPAIR SMOKE: PASS")

## 12. Output directory and immutable run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook13_Dream_Confirm"
    )
else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook13_Dream_Confirm"
    )

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

signature_payload = {
    "protocol_version":
        "dream-independent-selfrepair-confirm-v1",
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "model_id":
        MODEL_ID,
    "model_revision":
        MODEL_REVISION,
    "quantization":
        "bnb_nf4_4bit_exploratory",
    "actions":
        ACTIONS,
    "repair_seeds":
        REPAIR_SEEDS,
    "denoising_steps":
        DENOISING_STEPS,
    "temperature":
        TEMPERATURE,
    "top_p":
        TOP_P,
    "alg":
        ALG,
    "alg_temp":
        ALG_TEMP,
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

TRIALS_PATH = (
    OUT_DIR
    / "independent_self_repair_trials.jsonl"
)

MANIFEST_PATH = (
    OUT_DIR
    / "run_manifest.json"
)

if CLEAR_OLD_RESULTS:
    for p in [
        TRIALS_PATH,
        MANIFEST_PATH,
    ]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    if old.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 13 outputs use a different configuration."
        )

else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "run_signature":
                    RUN_SIGNATURE,
                **signature_payload,
                "gpu":
                    GPU_NAME,
                "gpu_total_gb":
                    GPU_GB,
                "torch":
                    torch.__version__,
                "transformers":
                    transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 13. Resume-safe 150-trial confirmation sweep

In [ ]:
def trial_id(
    failure_id,
    action,
    repair_seed,
):
    return (
        failure_id
        + "|confirm|"
        + action
        + "|"
        + str(repair_seed)
    )


EXPECTED_IDS = {
    trial_id(
        failure["failure_id"],
        action,
        repair_seed,
    )
    for failure in FAILURES
    for action in ACTIONS
    for repair_seed in REPAIR_SEEDS
}

assert len(EXPECTED_IDS) == 150


def load_completed(path):
    done = {}

    if not path.exists():
        return done

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        for line_number, line in enumerate(
            f,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"Malformed JSONL at line {line_number}."
                ) from e

            if rec.get("run_signature") != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found a trial from another Notebook 13 configuration."
                )

            tid = rec["trial_id"]

            if tid in done:
                raise RuntimeError(
                    f"Duplicate trial_id: {tid}"
                )

            done[tid] = rec

    return done


completed = load_completed(
    TRIALS_PATH
)

print(
    "Already completed:",
    len(completed),
    "/ 150",
)

counter = 0

with open(
    TRIALS_PATH,
    "a",
    encoding="utf-8",
) as fp:

    for failure in FAILURES:
        for action in ACTIONS:
            for repair_seed in REPAIR_SEEDS:
                counter += 1

                tid = trial_id(
                    failure["failure_id"],
                    action,
                    repair_seed,
                )

                if tid in completed:
                    print(
                        f"[{counter}/150] SKIP {tid}"
                    )
                    continue

                try:
                    rec = run_self_repair(
                        failure,
                        action,
                        repair_seed,
                    )

                    rec["trial_id"] = tid
                    rec["run_signature"] = RUN_SIGNATURE
                    rec["status"] = "ok"
                    rec["error"] = None

                except torch.cuda.OutOfMemoryError as e:
                    torch.cuda.empty_cache()
                    gc.collect()

                    rec = {
                        "trial_id": tid,
                        "failure_id": failure["failure_id"],
                        "problem_id": failure["problem_id"],
                        "dataset": failure["dataset"],
                        "confirmatory_role": failure["confirmatory_role"],
                        "action": action,
                        "repair_seed": repair_seed,
                        "status": "oom",
                        "error": str(e),
                        "run_signature": RUN_SIGNATURE,
                    }

                except Exception as e:
                    rec = {
                        "trial_id": tid,
                        "failure_id": failure["failure_id"],
                        "problem_id": failure["problem_id"],
                        "dataset": failure["dataset"],
                        "confirmatory_role": failure["confirmatory_role"],
                        "action": action,
                        "repair_seed": repair_seed,
                        "status": "error",
                        "error": f"{type(e).__name__}: {e}",
                        "run_signature": RUN_SIGNATURE,
                    }

                fp.write(
                    json.dumps(
                        rec,
                        ensure_ascii=False,
                    )
                )
                fp.write("\n")
                fp.flush()

                if rec["status"] == "ok":
                    print(
                        f"[{counter}/150] {tid} | "
                        f"task={rec['task_success']} | "
                        f"samewrong={rec['same_wrong_numeric']} | "
                        f"NFE={rec['actual_forward_calls']}"
                    )
                else:
                    print(
                        f"[{counter}/150] {tid} | "
                        f"{rec['status']} | {rec['error']}"
                    )

print("Independent confirmation sweep finished.")

## 14. Validate exactly 150 completed trials

In [ ]:
records = load_completed(
    TRIALS_PATH
)

missing = EXPECTED_IDS - set(records)
extra = set(records) - EXPECTED_IDS

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError(
        "Confirmation incomplete. Rerun the sweep cell; completed trials will be skipped."
    )

if extra:
    raise RuntimeError(
        "Unexpected Notebook 13 trial IDs exist."
    )

assert len(records) == 150

rows = [
    records[tid]
    for tid in sorted(EXPECTED_IDS)
]

status_counts = Counter(
    x["status"]
    for x in rows
)

print("Statuses:", dict(status_counts))

if status_counts.get("oom", 0):
    raise RuntimeError(
        "At least one confirmatory trial OOMed."
    )

if status_counts.get("error", 0):
    raise RuntimeError(
        "At least one confirmatory trial errored."
    )

TRIAL_DF = pd.DataFrame(rows)

assert len(TRIAL_DF) == 150
assert TRIAL_DF["trial_id"].nunique() == 150
assert (
    TRIAL_DF["actual_forward_calls"]
    >= DENOISING_STEPS
).all()

forced = TRIAL_DF[
    TRIAL_DF["forced_tokens"] > 0
]

assert (
    forced["token_position_preservation"]
    + 1e-12
    >= forced["forced_fraction"]
).all()

PRIMARY_DF = TRIAL_DF[
    TRIAL_DF["confirmatory_role"]
    == "primary_unseen_problem"
].copy()

SECONDARY_DF = TRIAL_DF[
    TRIAL_DF["confirmatory_role"]
    == "secondary_new_trajectory_same_problem"
].copy()

assert len(PRIMARY_DF) == 120
assert len(SECONDARY_DF) == 30

print("150-trial matrix integrity: PASS")
print("Primary unseen-problem trials:", len(PRIMARY_DF))
print("Secondary SVAMP trajectory trials:", len(SECONDARY_DF))

## 15. Primary unseen-problem action summary

In [ ]:
def action_summary(df):
    rows = []

    for action, group in df.groupby("action"):
        failed = group[
            ~group["task_success"]
        ]

        rows.append({
            "action":
                action,
            "trials":
                len(group),
            "task_success_rate":
                float(
                    group["task_success"].mean()
                ),
            "strict_success_rate":
                float(
                    group["strict_success"].mean()
                ),
            "numeric_success_rate":
                float(
                    group["numeric_success"].mean()
                ),
            "mean_preservation":
                float(
                    group["token_position_preservation"].mean()
                ),
            "mean_nfe":
                float(
                    group["actual_forward_calls"].mean()
                ),
            "same_wrong_given_failure":
                float(
                    failed["same_wrong_numeric"].mean()
                )
                if len(failed)
                else 0.0,
        })

    return pd.DataFrame(rows)


PRIMARY_ACTION_DF = action_summary(
    PRIMARY_DF
)

ALL_ACTION_DF = action_summary(
    TRIAL_DF
)

display(PRIMARY_ACTION_DF)

## 16. Secondary SVAMP trajectory summary

In [ ]:
SECONDARY_ACTION_DF = action_summary(
    SECONDARY_DF
)

display(SECONDARY_ACTION_DF)

## 17. Dataset × action summaries

In [ ]:
DATASET_DF = (
    TRIAL_DF.groupby(
        [
            "confirmatory_role",
            "dataset",
            "action",
        ],
        as_index=False,
    )
    .agg(
        trials=(
            "task_success",
            "size",
        ),
        task_success_rate=(
            "task_success",
            "mean",
        ),
        same_wrong_rate=(
            "same_wrong_numeric",
            "mean",
        ),
        mean_preservation=(
            "token_position_preservation",
            "mean",
        ),
        mean_nfe=(
            "actual_forward_calls",
            "mean",
        ),
    )
)

display(DATASET_DF)

## 18. Failure-cluster bootstrap CIs

The bootstrap resamples **failure IDs**, keeping all five repair seeds within each selected failure/action condition.

Primary CIs use only the 8 unseen problem IDs.

In [ ]:
def cluster_bootstrap_rate(
    df,
    action,
    value_col,
    n_boot=20000,
    seed=13001,
):
    sub = df[
        df["action"] == action
    ].copy()

    fids = sorted(
        sub["failure_id"].unique()
    )

    by_failure = {
        fid: sub[
            sub["failure_id"] == fid
        ][value_col].astype(float).to_numpy()
        for fid in fids
    }

    observed = float(
        sub[value_col].astype(float).mean()
    )

    rng = np.random.default_rng(seed)
    boots = np.empty(
        n_boot,
        dtype=float,
    )

    for i in range(n_boot):
        sampled = rng.choice(
            fids,
            size=len(fids),
            replace=True,
        )

        vals = np.concatenate(
            [
                by_failure[fid]
                for fid in sampled
            ]
        )

        boots[i] = float(
            vals.mean()
        )

    lo, hi = np.quantile(
        boots,
        [0.025, 0.975],
    )

    return (
        observed,
        float(lo),
        float(hi),
    )


ci_rows = []

for idx, action in enumerate(ACTIONS):
    est, lo, hi = cluster_bootstrap_rate(
        PRIMARY_DF,
        action,
        "task_success",
        seed=13001 + idx,
    )

    ci_rows.append({
        "action":
            action,
        "task_success":
            est,
        "ci95_low":
            lo,
        "ci95_high":
            hi,
    })

PRIMARY_CI_DF = pd.DataFrame(
    ci_rows
)

display(PRIMARY_CI_DF)

## 19. Paired primary contrasts

All comparisons use exactly matched failure IDs and repair seeds.

Two success contrasts are reported:

- `open_last_75` vs `global`;
- `open_last_25` vs `global`.

A one-sided exact discordant-pair test asks whether the selective scope wins more often than global. A two-sided exact test is also reported so a negative result is visible rather than hidden.

In [ ]:
def paired_success_contrast(
    df,
    selective_action,
):
    left = (
        df[
            df["action"]
            == selective_action
        ][
            [
                "failure_id",
                "repair_seed",
                "task_success",
            ]
        ]
        .rename(
            columns={
                "task_success":
                    "selective_success"
            }
        )
    )

    right = (
        df[
            df["action"]
            == "global"
        ][
            [
                "failure_id",
                "repair_seed",
                "task_success",
            ]
        ]
        .rename(
            columns={
                "task_success":
                    "global_success"
            }
        )
    )

    paired = left.merge(
        right,
        on=[
            "failure_id",
            "repair_seed",
        ],
        how="inner",
    )

    s = paired[
        "selective_success"
    ].astype(int).to_numpy()

    g = paired[
        "global_success"
    ].astype(int).to_numpy()

    selective_only = int(
        np.sum(
            (s == 1)
            & (g == 0)
        )
    )

    global_only = int(
        np.sum(
            (s == 0)
            & (g == 1)
        )
    )

    discordant = (
        selective_only
        + global_only
    )

    if discordant == 0:
        p_one = 1.0
        p_two = 1.0
    else:
        p_one = float(
            binomtest(
                selective_only,
                discordant,
                p=0.5,
                alternative="greater",
            ).pvalue
        )

        p_two = float(
            binomtest(
                selective_only,
                discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )

    return {
        "selective_action":
            selective_action,
        "n_paired_trials":
            len(paired),
        "selective_task_success":
            float(s.mean()),
        "global_task_success":
            float(g.mean()),
        "difference_vs_global":
            float(
                s.mean()
                - g.mean()
            ),
        "selective_only":
            selective_only,
        "global_only":
            global_only,
        "one_sided_p_selective_gt_global":
            p_one,
        "two_sided_p":
            p_two,
    }


PRIMARY_CONTRAST_DF = pd.DataFrame(
    [
        paired_success_contrast(
            PRIMARY_DF,
            "open_last_75",
        ),
        paired_success_contrast(
            PRIMARY_DF,
            "open_last_25",
        ),
    ]
)

display(PRIMARY_CONTRAST_DF)

## 20. Preservation–recurrence confirmation

Notebook 12 suggested that narrow repair can preserve more text while remaining trapped in the same wrong numeric state.

The confirmatory descriptive quantities are:

- preservation gap: `open_last_25 - global`;
- recurrence gap among still-failed repairs: `open_last_25 - global`;
- recurrence reduction from widening the reopened region: `open_last_25 - open_last_75`.

These are reported without forcing a significance claim.

In [ ]:
primary_lookup = {
    row["action"]: row
    for row in PRIMARY_ACTION_DF.to_dict(
        "records"
    )
}

PRESERVATION_GAP = (
    primary_lookup[
        "open_last_25"
    ][
        "mean_preservation"
    ]
    -
    primary_lookup[
        "global"
    ][
        "mean_preservation"
    ]
)

RECURRENCE_GAP_NARROW_VS_GLOBAL = (
    primary_lookup[
        "open_last_25"
    ][
        "same_wrong_given_failure"
    ]
    -
    primary_lookup[
        "global"
    ][
        "same_wrong_given_failure"
    ]
)

RECURRENCE_REDUCTION_WIDER = (
    primary_lookup[
        "open_last_25"
    ][
        "same_wrong_given_failure"
    ]
    -
    primary_lookup[
        "open_last_75"
    ][
        "same_wrong_given_failure"
    ]
)

TRADEOFF_SUMMARY = {
    "primary_preservation_gap_open25_minus_global":
        float(
            PRESERVATION_GAP
        ),
    "primary_same_wrong_gap_open25_minus_global":
        float(
            RECURRENCE_GAP_NARROW_VS_GLOBAL
        ),
    "primary_same_wrong_reduction_open25_minus_open75":
        float(
            RECURRENCE_REDUCTION_WIDER
        ),
}

print(
    json.dumps(
        TRADEOFF_SUMMARY,
        indent=2,
    )
)

## 21. Per-failure scope heterogeneity and confirmatory ORH

This analysis uses only the **8 primary unseen problem IDs**.

If the best action differs across failures, that supports instance-dependent repairability even when no single selective scope beats global overall.

In [ ]:
surface_rows = []

for (
    failure_id,
    problem_id,
    dataset,
    action,
), group in PRIMARY_DF.groupby(
    [
        "failure_id",
        "problem_id",
        "dataset",
        "action",
    ]
):
    assert len(group) == 5

    surface_rows.append({
        "failure_id":
            failure_id,
        "problem_id":
            problem_id,
        "dataset":
            dataset,
        "action":
            action,
        "task_success_rate":
            float(
                group["task_success"].mean()
            ),
        "same_wrong_rate":
            float(
                group["same_wrong_numeric"].mean()
            ),
        "mean_preservation":
            float(
                group[
                    "token_position_preservation"
                ].mean()
            ),
        "mean_nfe":
            float(
                group["actual_forward_calls"].mean()
            ),
    })

PRIMARY_SURFACE_DF = pd.DataFrame(
    surface_rows
)

assert len(PRIMARY_SURFACE_DF) == 24

fixed = (
    PRIMARY_SURFACE_DF.groupby(
        "action",
        as_index=False,
    )
    .agg(
        mean_task_success=(
            "task_success_rate",
            "mean",
        ),
        mean_nfe=(
            "mean_nfe",
            "mean",
        ),
        mean_preservation=(
            "mean_preservation",
            "mean",
        ),
    )
    .sort_values(
        [
            "mean_task_success",
            "mean_nfe",
            "mean_preservation",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    )
)

best_fixed = fixed.iloc[0]

oracle_rows = []

for failure_id, group in PRIMARY_SURFACE_DF.groupby(
    "failure_id"
):
    best = group.sort_values(
        [
            "task_success_rate",
            "mean_nfe",
            "mean_preservation",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    ).iloc[0]

    oracle_rows.append({
        "failure_id":
            failure_id,
        "problem_id":
            best["problem_id"],
        "dataset":
            best["dataset"],
        "oracle_action":
            best["action"],
        "oracle_task_success":
            float(
                best["task_success_rate"]
            ),
    })

PRIMARY_ORACLE_DF = pd.DataFrame(
    oracle_rows
)

best_fixed_mean = float(
    best_fixed["mean_task_success"]
)

oracle_mean = float(
    PRIMARY_ORACLE_DF[
        "oracle_task_success"
    ].mean()
)

PRIMARY_ORH = float(
    oracle_mean
    - best_fixed_mean
)

oracle_distribution = (
    PRIMARY_ORACLE_DF[
        "oracle_action"
    ]
    .value_counts(
        normalize=True
    )
    .to_dict()
)

print(
    "Primary best fixed:",
    best_fixed["action"],
)

print(
    "Primary best fixed success:",
    round(best_fixed_mean, 4),
)

print(
    "Primary oracle success:",
    round(oracle_mean, 4),
)

print(
    "Primary ORH:",
    round(PRIMARY_ORH, 4),
)

print(
    "Primary oracle action distribution:",
    oracle_distribution,
)

display(PRIMARY_ORACLE_DF)

## 22. Independent confirmation gate

The gate is deliberately modest and does **not** require selective remasking to beat global regeneration.

Confirmation passes when:

1. all 150 trials complete;
2. all 120 primary unseen-problem trials are valid;
3. mechanical token preservation is valid;
4. at least 8 task-successful repairs occur on the primary unseen problems;
5. both primary datasets show at least one successful repair;
6. at least two of the three tested actions succeed on the primary set;
7. primary per-failure oracle headroom is positive **or** at least two different actions are selected as oracle actions across the 8 failures.

This tests reproducibility of **self-repair and scope heterogeneity**, not superiority of a particular scope.

In [ ]:
primary_successes = int(
    PRIMARY_DF[
        "task_success"
    ].sum()
)

primary_datasets_with_success = int(
    PRIMARY_DF.loc[
        PRIMARY_DF[
            "task_success"
        ],
        "dataset",
    ].nunique()
)

primary_actions_with_success = int(
    PRIMARY_DF.loc[
        PRIMARY_DF[
            "task_success"
        ],
        "action",
    ].nunique()
)

primary_oracle_actions = int(
    PRIMARY_ORACLE_DF[
        "oracle_action"
    ].nunique()
)

mechanical_preservation_ok = bool(
    (
        forced[
            "token_position_preservation"
        ]
        + 1e-12
        >=
        forced[
            "forced_fraction"
        ]
    ).all()
)

CONFIRM_GATE = {
    "all_trials":
        int(
            len(TRIAL_DF)
        ),
    "unique_trial_ids":
        int(
            TRIAL_DF[
                "trial_id"
            ].nunique()
        ),
    "primary_unseen_problem_trials":
        int(
            len(PRIMARY_DF)
        ),
    "secondary_svamp_trials":
        int(
            len(SECONDARY_DF)
        ),
    "primary_task_successful_repairs":
        primary_successes,
    "primary_datasets_with_success":
        primary_datasets_with_success,
    "primary_actions_with_success":
        primary_actions_with_success,
    "primary_best_fixed_action":
        str(
            best_fixed["action"]
        ),
    "primary_best_fixed_task_success":
        best_fixed_mean,
    "primary_oracle_task_success":
        oracle_mean,
    "primary_oracle_repair_headroom":
        PRIMARY_ORH,
    "primary_oracle_actions_used":
        primary_oracle_actions,
    "primary_preservation_gap_open25_minus_global":
        float(
            PRESERVATION_GAP
        ),
    "primary_same_wrong_gap_open25_minus_global":
        float(
            RECURRENCE_GAP_NARROW_VS_GLOBAL
        ),
    "primary_same_wrong_reduction_open25_minus_open75":
        float(
            RECURRENCE_REDUCTION_WIDER
        ),
    "mechanical_preservation_ok":
        mechanical_preservation_ok,
    "minimum_actual_nfe":
        int(
            TRIAL_DF[
                "actual_forward_calls"
            ].min()
        ),
    "quantization":
        "bnb_nf4_4bit_exploratory",
}

CONFIRM_GATE[
    "DREAM_SELF_REPAIR_INDEPENDENTLY_CONFIRMED"
] = bool(
    CONFIRM_GATE[
        "all_trials"
    ] == 150
    and CONFIRM_GATE[
        "unique_trial_ids"
    ] == 150
    and CONFIRM_GATE[
        "primary_unseen_problem_trials"
    ] == 120
    and CONFIRM_GATE[
        "mechanical_preservation_ok"
    ]
    and CONFIRM_GATE[
        "minimum_actual_nfe"
    ] >= DENOISING_STEPS
    and CONFIRM_GATE[
        "primary_task_successful_repairs"
    ] >= 8
    and CONFIRM_GATE[
        "primary_datasets_with_success"
    ] == 2
    and CONFIRM_GATE[
        "primary_actions_with_success"
    ] >= 2
    and (
        CONFIRM_GATE[
            "primary_oracle_repair_headroom"
        ] > 0
        or CONFIRM_GATE[
            "primary_oracle_actions_used"
        ] >= 2
    )
)

print(
    json.dumps(
        CONFIRM_GATE,
        indent=2,
    )
)

## 23. Save Notebook 13 artifacts

In [ ]:
cohort_path = (
    OUT_DIR
    / "confirmation_cohort.csv"
)

primary_action_path = (
    OUT_DIR
    / "primary_action_summary.csv"
)

secondary_action_path = (
    OUT_DIR
    / "secondary_svamp_summary.csv"
)

dataset_path = (
    OUT_DIR
    / "dataset_action_summary.csv"
)

ci_path = (
    OUT_DIR
    / "primary_action_ci.csv"
)

contrast_path = (
    OUT_DIR
    / "primary_selective_vs_global.csv"
)

surface_path = (
    OUT_DIR
    / "primary_repair_surface.csv"
)

oracle_path = (
    OUT_DIR
    / "primary_oracle.csv"
)

tradeoff_path = (
    OUT_DIR
    / "preservation_recurrence_tradeoff.json"
)

gate_path = (
    OUT_DIR
    / "DREAM_INDEPENDENT_CONFIRMATION_GATE.json"
)

pd.DataFrame(
    [
        {
            "failure_id":
                x["failure_id"],
            "problem_id":
                x["problem_id"],
            "dataset":
                x["dataset"],
            "source_generation_seed":
                x["seed"],
            "confirmatory_role":
                x["confirmatory_role"],
        }
        for x in FAILURES
    ]
).to_csv(
    cohort_path,
    index=False,
)

PRIMARY_ACTION_DF.to_csv(
    primary_action_path,
    index=False,
)

SECONDARY_ACTION_DF.to_csv(
    secondary_action_path,
    index=False,
)

DATASET_DF.to_csv(
    dataset_path,
    index=False,
)

PRIMARY_CI_DF.to_csv(
    ci_path,
    index=False,
)

PRIMARY_CONTRAST_DF.to_csv(
    contrast_path,
    index=False,
)

PRIMARY_SURFACE_DF.to_csv(
    surface_path,
    index=False,
)

PRIMARY_ORACLE_DF.to_csv(
    oracle_path,
    index=False,
)

tradeoff_path.write_text(
    json.dumps(
        TRADEOFF_SUMMARY,
        indent=2,
    ),
    encoding="utf-8",
)

gate_path.write_text(
    json.dumps(
        CONFIRM_GATE,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved Notebook 13 outputs:",
    OUT_DIR,
)

## 24. Artifact manifest

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


artifact_paths = [
    MANIFEST_PATH,
    TRIALS_PATH,
    cohort_path,
    primary_action_path,
    secondary_action_path,
    dataset_path,
    ci_path,
    contrast_path,
    surface_path,
    oracle_path,
    tradeoff_path,
    gate_path,
]

artifact_manifest = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "run_signature":
        RUN_SIGNATURE,
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "protocol_version":
        "dream-independent-selfrepair-confirm-v1",
    "files": {
        p.name:
            sha256_file(p)
        for p in artifact_paths
        if p.exists()
    },
}

artifact_manifest_path = (
    OUT_DIR
    / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        artifact_manifest,
        indent=2,
    )
)

## 25. ZIP and download all results

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook13_Dream_Independent_Confirmation_results"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUT_DIR,
)

print("Created:", zip_path)

print(
    "Size MB:",
    round(
        Path(
            zip_path
        ).stat().st_size
        / 1024**2,
        3,
    ),
)

if DOWNLOAD_RESULTS:
    from google.colab import files

    files.download(
        zip_path
    )

# Progression rule

If `DREAM_SELF_REPAIR_INDEPENDENTLY_CONFIRMED = true`, stop expanding the quantized Dream evidence.

The next priority becomes **model generalization**:

1. run the same selective-remasking concept on a second genuine DLM;
2. then obtain at least one full-precision replication on a larger-memory GPU;
3. after that, freeze the central RepairScope-X tables and figures.

A null selective-vs-global result is acceptable. The central claim is about **per-instance repairability, scope-dependent preservation, and error recurrence**, not that selective repair must always outperform fresh regeneration.